# GPU essentials: what a GPU is, and why climate models use them

*The mental model — SIMD, threads and blocks, saturation — and one portable kernel.*

The realistic simulations later this week — a global ocean, a coupled ocean–sea ice Arctic — do not run
on a laptop: they run on **GPUs**. This notebook builds the *mental model* you need to read a model
configuration and understand why it asks for `architecture = GPU()`, and when that choice pays off. We
are deliberately *not* teaching GPU programming in depth — we want intuition, with just enough runnable
code to watch the ideas move.

Everything falls back to the CPU, so the notebook runs top-to-bottom on any laptop. If a GPU is present
— NVIDIA through CUDA, or an Apple chip through Metal — the same cells use it.

In [38]:
using InteractiveUtils      
using CairoMakie, BenchmarkTools
using CUDA              
using Metal
using KernelAbstractions
import KernelAbstractions: @kernel, @index, get_backend, synchronize

## Why a GPU? two different shapes of hardware

A CPU and a GPU answer two different questions. A **CPU** has a handful of fast, complex cores built to finish *one* chain of instructions as quickly as possible — each clever and quick. A **GPU** has *thousands* of simple, slower cores built to do the *same* operation on *mountains* of data at once — an army of students, each doing a little, all in step.

| | CPU | GPU |
|---|---|---|
| cores | a few (tens), fast | thousands, slow |
| optimized for | latency (one task, fast) | throughput (a million tasks) |
| best at | branchy, sequential logic | the same operation over a huge array |
| memory | large, deep caches | smaller, very high bandwidth |

The price of that throughput: the cores run in lockstep (so branches that diverge are expensive), the
data must be *copied* to the device first, and the win appears only when there is enough parallel work to
keep the army busy. A climate model — the same stencil applied to every cell of a huge grid — is almost
the perfect fit.

## One instruction, many data: SIMD

The GPU idea is not foreign to the CPU; it is the CPU idea taken to an extreme. Even a single CPU core
does **SIMD** — *Single Instruction, Multiple Data* — applying one instruction to a small vector of
numbers (a few "lanes") at once. We can see it: an element-wise loop marked `@simd` compiles to *packed*
vector instructions instead of one-number-at-a-time scalar ones. We use `y = a·x + y`, the very update
the GPU kernel below will run:

In [24]:
function simd_saxpy!(y, a, x)
    @inbounds @simd for i in eachindex(y, x)
        y[i] = a * x[i] + y[i]
    end
    return y
end

@code_llvm debuginfo=:none simd_saxpy!(zeros(256), 2.0, rand(256))

; Function Signature: simd_saxpy!(Array{Float64, 1}, Float64, Array{Float64, 1})
define nonnull ptr @"julia_simd_saxpy!_48865"(ptr noundef nonnull align 8 dereferenceable(24) %"y::Array", double %"a::Float64", ptr noundef nonnull align 8 dereferenceable(24) %"x::Array") #0 {
top:
  %jlcallframe1 = alloca [3 x ptr], align 8
  %gcframe2 = alloca [4 x ptr], align 16
  call void @llvm.memset.p0.i64(ptr align 16 %gcframe2, i8 0, i64 32, i1 true)
  %pgcstack = call ptr inttoptr (i64 4304060188 to ptr)(i64 4304060224) #11
  store i64 8, ptr %gcframe2, align 8
  %task.gcstack = load ptr, ptr %pgcstack, align 8
  %frame.prev = getelementptr inbounds ptr, ptr %gcframe2, i64 1
  store ptr %task.gcstack, ptr %frame.prev, align 8
  store ptr %gcframe2, ptr %pgcstack, align 8
  %"y::Array.size_ptr" = getelementptr inbounds i8, ptr %"y::Array", i64 16
  %"y::Array.size.0.copyload" = load i64, ptr %"y::Array.size_ptr", align 8
  %"x::Array.size_ptr" = getelementptr inbounds i8, ptr %"x::Array", i64 16

Look for the `<2 x double>` (or `<4 x float>`) types in that IR: the compiler is doing the multiply-add on
*two lanes at a time*. A CPU core has a few such lanes; a GPU has the same idea with **thousands** of
them, spread across its cores — NVIDIA calls the grouped-lane execution *SIMT* (Single Instruction,
Multiple Threads). That is the whole leap: from a handful of lanes to thousands.

## Choosing a device, once

A model selects its hardware in a single place; we do the same. Prefer a CUDA GPU, then an Apple GPU via
Metal, otherwise the CPU. GPUs compute in **single precision**, so device arrays are `Float32`.

In [25]:
backend, device_name = if CUDA.functional()
    CUDA.CUDABackend(), "CUDA GPU"
elseif Metal.functional()
    Metal.MetalBackend(), "Apple GPU (Metal)"
else
    KernelAbstractions.CPU(), "CPU"
end

@info "Running on the $device_name"

[ Info: Running on the Apple GPU (Metal)


## Array programming: parallelism for free

You rarely start by writing kernels. A GPU array is a *different type* that supports the *same*
operations as an ordinary one — so once the data is on the device, broadcasts and reductions already run
there, in parallel, with no kernel in sight. Move a field across, then use the usual array syntax:

In [37]:
xₕ = randn(Float32, 512, 512)                          # a field on the host (CPU)
x  = KernelAbstractions.zeros(backend, Float32, 512, 512)
copyto!(x, xₕ)                                         # the same data, now on the device

∇²x = circshift(x, (1, 0)) .+ circshift(x, (-1, 0)) .+
      circshift(x, (0, 1)) .+ circshift(x, (0, -1)) .- 4 .* x   # a Laplacian stencil, run on the device
(sum(x), maximum(abs, ∇²x))                            # reductions run on the device too

(120.31627f0, 20.31582f0)

Not one line of that mentions threads or kernels — Julia's broadcasting (the dots) plus the GPU array
type did the parallelism. A great deal of model code is exactly this: fused broadcasts and reductions
over fields, fast on a GPU for free. Kernels, next, are the escape hatch — for the loop that *no* built-in
array operation expresses.

## Threads and blocks: a kernel is a loop over threads

How do you drive thousands of lanes without writing thousands of loops? You write the **body** of the
loop *once* — a *kernel* — and the hardware runs one copy per **thread**, each on its own element. A
serial CPU loop and its kernel are the same arithmetic, reorganized:

```julia
for i in 1:N             #  ↔   i = @index(Global)         # the kernel is the *inside* of the loop;
    y[i] = a*x[i] + y[i] #           y[i] = a*x[i] + y[i]   # the hardware supplies the loop — one i per thread
end
```

Threads are organized into **blocks** (KernelAbstractions calls them *workgroups*) — the launch size,
say 256 threads each — which the hardware schedules across its cores:

```
  block 0                block 1                ...
  ┌────────────────────┐ ┌────────────────────┐
  │ t0 t1 t2 … t255    │ │ t0 t1 t2 … t255    │   each tₖ runs the kernel body on one element
  └────────────────────┘ └────────────────────┘
```

Here is the kernel form of `y = a·x + y` (a "saxpy") — the same update we just watched the CPU vectorize,
now with one thread per element — written with
[KernelAbstractions.jl](https://github.com/JuliaGPU/KernelAbstractions.jl):

In [27]:
@kernel function _saxpy!(y, a, x)
    i = @index(Global)                  # this thread's element — the loop index the hardware hands us
    @inbounds y[i] = a * x[i] + y[i]
end

function saxpy!(y, a, x)
    backend = get_backend(y)            # CPU(), CUDABackend(), or MetalBackend() — read off the data
    _saxpy!(backend, 256)(y, a, x; ndrange = length(y))   # 256 threads per block, length(y) total
    synchronize(backend)
    return y
end

saxpy! (generic function with 1 method)

On a plain CPU array it is just an (optionally multithreaded) loop:

In [28]:
let x = ones(Float32, 1000), y = ones(Float32, 1000)
    saxpy!(y, 2f0, x)
    y[1]                                # 2*1 + 1 = 3
end

3.0f0

## Saturation: a GPU needs enough work

Thousands of cores only help if there is enough work to spread across them. Below a certain size the
launch overhead dominates and most of the army stands idle; throughput climbs with the problem size and
then *saturates* at the hardware's ceiling. We measure it directly — elements updated per second by
`saxpy!` as the array grows — on the chosen device:

In [ ]:
function throughput(backend, n)
    x = KernelAbstractions.zeros(backend, Float32, n); fill!(x, 1f0)
    y = KernelAbstractions.zeros(backend, Float32, n); fill!(y, 1f0)
    saxpy!(y, 2f0, x)                                    # warm up: compile the kernel
    t = @belapsed (saxpy!($y, 2f0, $x); synchronize($backend))
    return n / t                                         # elements per second
end

sizes     = [2^k for k in 8:2:24]                        # ~250 up to ~16 million elements
cpu_rates = [throughput(KernelAbstractions.CPU(), n) for n in sizes]
gpu_rates = backend isa KernelAbstractions.CPU ? nothing : [throughput(backend, n) for n in sizes]
nothing #hide

Plotting throughput against array size makes the saturation visible:

In [ ]:
plot_title = gpu_rates === nothing ? "saxpy throughput (CPU only)" : "saxpy throughput: CPU vs $device_name"
fig = Figure(size = (760, 420))
ax = Axis(fig[1, 1]; xlabel = "array size (elements)", ylabel = "throughput (elements / s)",
          xscale = log10, yscale = log10, title = plot_title)
scatterlines!(ax, sizes, cpu_rates; linewidth = 3, markersize = 12, label = "CPU")
gpu_rates === nothing || scatterlines!(ax, sizes, gpu_rates; linewidth = 3, markersize = 12, label = device_name)
axislegend(ax; position = :rb)
save("gpu_saturation.png", fig)
nothing #hide

The two curves tell the whole story. On a CPU throughput is fairly flat — a few cores hit their modest
peak almost at once, so the *smallest* arrays even run faster there. The GPU starts *below* the CPU, with
thousands of cores sitting idle, but as the arrays grow it climbs, crosses over, and pulls far ahead —
plateauing at a much higher ceiling once it saturates. The practical rule follows directly: **a GPU pays
off for large, parallel problems**; starved of work, it is genuinely slower than a CPU.

## One kernel, every device

Notice what did *not* change between a CPU run and a GPU run: `_saxpy!`. The same kernel body, the same
`saxpy!` launcher — only the array *type* differed, and `get_backend` read the device off it. This is
exactly how the models you will run are written: one set of kernels, dispatched onto whatever
`architecture` the configuration names —

```julia
architecture = CPU()      # develop and debug on a laptop
architecture = GPU()      # the identical model, now on the device
```

That is the takeaway. You will not *write* GPU code this week, but you now know what it is doing and why
it is there: thousands of lanes, fed enough data to stay busy, running the same loop body a CPU would —
just very many more of them, all at once.

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*